In [1]:
%%time
import numpy as np
import matplotlib.pyplot as plt
import sys
import math
import pandas as pd
import numpy as np

# https://pypi.org/project/era5cli/

CPU times: user 1.24 s, sys: 1.31 s, total: 2.55 s
Wall time: 1.39 s


In [2]:
import xarray as xr
ds = xr.open_dataset("../../data/data_stream-oper_stepType-instant.nc", engine='h5netcdf')

In [3]:
# Project imports
from sutton import (
    Params,
    thomas,
    our_central_difference,
    integrate_T_implicit,
    integrate_H2O_implicit,
    stability,
    saturation_vapor_pressure,
    vapor_concentration_RH,
)
print("sutton imports ready")

sutton imports ready


In [4]:
lat = ds['latitude'].values
lon = ds['longitude'].values

# Coordinates of interest
lat_point = 45.6667
lon_point = -119.6667

# Find the closest indices
lat_idx = (np.abs(lat - lat_point)).argmin() + 1
lon_idx = (np.abs(lon - lon_point)).argmin() + 1

# Extract data at the closest point
data_at_point = ds.isel(latitude=lat_idx, longitude=lon_idx)

In [5]:
def get_lm(k,h,z, option = 'kz'):
    """
    mixing length option
    """
    d = 2/3*h
    
    if option == 'kz':
        lm = k*z
    else:
        a = 1
        lm = k*z   # k*a*(h/3)
        lm[z < a*(h - d)] = k*a*h/3 
    
        # lm = k*(z - d)    
        # lm[z < h] = k*h/3      
    return lm



In [6]:
def our_central_difference(s, dz):
    """
    Computes gradients of a data vector using central differencing. At edges, 
    forward and  backward differences are used
    """
    m = len(s)
    dsdz = np.zeros(m)
    
    # Central difference for interior points
    dsdz[1:m-1] = (s[2:m] - s[0:m-2]) / (2 * dz)
    #dsdz[1:m-1] = (s[2:m] - s[1:m-1]) / ( dz)
    
    # Forward difference at the first node
    dsdz[0] = (s[1] - s[0]) / dz
    
    # Backward difference at the last node
    dsdz[m-1] = (s[m-1] - s[m-2]) / dz
    
    return dsdz

In [7]:
# from dataclasses import dataclass, asdict, field
# import numpy as np
# from functools import cached_property
# from sutton import saturation_vapor_pressure, vapor_concentration_RH  
# from typing import Optional


# @dataclass
# class Params:
#     # --- core physics/numerics  ---
#     k: float = 0.4
#     zom_f: float = 0.005           # m (wet/downwind momentum roughness)
#     zom_c: float = 0.05            # m (dry/upwind momentum roughness)

#     h: float = 2.3                 # m 
#     Lx: float = 2000.0             # m
#     Hmax: float = 100.0            # m
#     dz: float = 0.5                # m
#     dx: float = 5.0                # m
#     xmin: float = 0.0              # m
        
#     ustar_f: Optional[float] = 0.15
#     ustar_c: Optional[float] = 0.15
        
#     fallow_fraction: float = 0.5
#     fallow_length: float = 1000.0  # m

#     # temperatures in °C; RH in %
#     T_sc: float = 30.0
#     T_sf: float = 50.0
#     T_a:  float = 30.0
#     RH_c: float = 43.0
#     RH_f: float = 20.0
#     RH_a: float = 18.0

#     # radiation (if actually use these in RN; Rao base case used RN-G as a given)
#     SW_in: float = 400.0
    
#     # emissivities
#     epsilon_f: float = 0.9
#     epsilon_c: float = 0.95
#     epsilon_a: float = 0.80
        
#     alpha_c: float = 0.22
#     alpha_f: float = 0.22

#     lm_option: str | None = "kz"

#     # reference wind for u* inference ( 4 m / 3.76 m/s)
#     U_ref: float = 3.76            # m s^-1 at z_ref_wind
#     z_ref_wind: float = 4.0        # m

#     # upstream H (W m^-2) if use Rao base-case RN-G = H01
#     H_f: float = 8.5*4.184e-3/1e-4
#     # downstream H (W m^-2) 
#     H_c: float = 0

#     # upstream LE (W m^-2)
#     LE_f: float = 0
#     LE_c: float = 8.5*4.184e-3/1e-4      
                

#     # ---- grids (properties) ----
#     @cached_property
#     def zmin(self) -> float:
#         # use momentum roughness as the lower plotting/solution bound
#         return self.zom_f

#     @cached_property
#     def z(self) -> np.ndarray:
#         zmax = self.Hmax
#         return np.arange(self.zmin + self.dz, zmax + self.dz, self.dz)

#     @cached_property
#     def x(self) -> np.ndarray:
#         xmax = self.Lx
#         return np.arange(self.xmin, xmax + self.dx, self.dx)

#     @property
#     def nz(self) -> int:
#         return len(self.z)

#     @property
#     def nx(self) -> int:
#         return len(self.x)

#     def resolve_ustars(self):
        
#         if self.ustar_f is None:
#             assert self.z_ref_wind > self.zom_f > 0
#             self.ustar_f = self.k * self.U_ref / np.log(self.z_ref_wind / self.zom_f)
        
#         if self.ustar_c is None:
#             assert self.z_ref_wind > self.zom_c > 0
#             self.ustar_c = self.k * self.U_ref / np.log(self.z_ref_wind / self.zom_c)
        
#         return self
    
#     # ---- block sizes (properties) ----
#     @cached_property
#     def fallow_size(self) -> int:
#         return int(self.fallow_length / self.dx)

#     @cached_property
#     def field_size(self) -> int:
#         return int(self.fallow_size * (1 - self.fallow_fraction) / self.fallow_fraction) \
#             if self.fallow_fraction > 0 else 0

#     # ---- humidity/thermo helpers (absolute humidity E in g m^-3) ----
#     @staticmethod
#     def _K(Tc: float) -> float:
#         return Tc + 273.15

#     @cached_property
#     def es_c(self) -> float:
#         # saturation vapor pressure at the dry surface (Pa)
#         return saturation_vapor_pressure(self._K(self.T_sc))

#     @cached_property
#     def es_f(self) -> float:
#         # saturation vapor pressure at the wet surface (Pa)
#         return saturation_vapor_pressure(self._K(self.T_sf))

#     @cached_property
#     def Q_c(self) -> float:
#         # absolute humidity at dry surface (g m^-3)
#         return vapor_concentration_RH(self.T_sc, self.RH_c)

#     @cached_property
#     def Q_f(self) -> float:
#         # absolute humidity at wet surface (g m^-3)
#         return vapor_concentration_RH(self.T_sf, self.RH_f)

#     @cached_property
#     def Q_a(self) -> float:
#         # absolute humidity at reference level (g m^-3)
#         return vapor_concentration_RH(self.T_a, self.RH_a)

#     # ---- optional compatibility: single dict for legacy call sites ----
#     def to_dict(self) -> dict:
#         d = asdict(self)
#         # attach derived fields and grids
#         d.update({
#             "x": self.x, "z": self.z, "nx": self.nx, "nz": self.nz,
#             "zmin": self.zmin, "zmax": self.Hmax, "xmax": self.Lx,
#             "ustar_f": self.ustar_f, "ustar_c": self.ustar_c,
#             "fallow_size": self.fallow_size, "field_size": self.field_size,
#             "Q_c": self.Q_c, "Q_f": self.Q_f, "Q_a": self.Q_a,
#             # 
#             "es_c": self.es_c, "es_f": self.es_f,
#         })
#         return d

ustar_f = 0.14
shear_ratio = 2


p = Params()
p.ustar_f = ustar_f
p.ustar_c = ustar_f*np.sqrt(shear_ratio)
p.ustar_c

0.19798989873223333

In [8]:
# ===================== PARAMS (consolidated) =====================
from dataclasses import dataclass, asdict
from typing import Optional
import numpy as np
from functools import cached_property
from sutton import saturation_vapor_pressure, vapor_concentration_RH
import sympy as sp

sigma_sb = 5.670374419e-8  # W m^-2 K^-4

@dataclass
class Params:
    # --- core physics/numerics  ---
    k: float = 0.4
    zom_f: float = 0.005           # m (wet/downwind momentum roughness)
    zom_c: float = 0.05            # m (dry/upwind momentum roughness)

    # CHANGE 5: canopy & displacement OPTIONAL inputs (per patch)
    # (If None → derive from z0m via alpha_m, disp_frac; see properties below)
    h_f_opt: Optional[float] = None
    h_c_opt: Optional[float] = None
    d_f_opt: Optional[float] = None
    d_c_opt: Optional[float] = None
    alpha_m: float = 0.10          # z0m ≈ alpha_m * h
    disp_frac: float = 0.67        # d ≈ disp_frac * h
    
    G : float = 0
    # domain/grid
    Lx: float = 2000.0             # m
    Hmax: float = 100.0            # m
    dz: float = 0.5                # m
    dx: float = 5.0                # m
    xmin: float = 0.0              # m

    # friction velocities (optional; None ⇒ compute later or via resolve_ustars)
    ustar_f: Optional[float] = 0.15
    ustar_c: Optional[float] = 0.15

    # block pattern
    fallow_fraction: float = 0.5
    fallow_length: float = 1000.0  # m

    # temperatures in °C; RH in %
    T_sc: float = 30.0
    T_sf: float = 50.0
    T_a:  float = 30.0
    RH_c: float = 43.0
    RH_f: float = 20.0
    RH_a: float = 18.0

    # radiation inputs (if you actually use RN components)
    SW_in: float = 400.0
    epsilon_f: float = 0.90
    epsilon_c: float = 0.95
    epsilon_a: float = 0.80
    alpha_c: float = 0.22
    alpha_f: float = 0.22

    lm_option: str | None = "kz"

    # reference wind for u* inference (if you compute from log law)
    U_ref: float = 3.76            # m s^-1 at z_ref_wind
    z_ref_wind: float = 4.0        # m

    # Upstream/base flux: Rao base case uses (RN-G)_up = H_f
    H_f: float = 8.5*4.184e-3/1e-4   # ≈ 356 W m^-2
    LE_f: float = 0.0                # upstream LE if you track it

    # avail_ratio 1: downstream available energy & partition 
    avail_ratio: float = 1   # (RN-G)_down / (RN-G)_up, dimensionless
    le_factor: float = 1.0             # 0..1: downstream LE_c = le_factor * avail_ratio

    # ----------------- helpers & grids -----------------
    @staticmethod
    def _K(Tc: float) -> float:
        return Tc + 273.15

    # derived canopy height & displacement (per patch)
    @cached_property
    def h_f(self) -> float:
        if self.h_f_opt is not None: return float(self.h_f_opt)
        if self.d_f_opt is not None: return float(self.d_f_opt) / self.disp_frac
        return float(self.zom_f) / self.alpha_m

    @cached_property
    def h_c(self) -> float:
        if self.h_c_opt is not None: return float(self.h_c_opt)
        if self.d_c_opt is not None: return float(self.d_c_opt) / self.disp_frac
        return float(self.zom_c) / self.alpha_m

    @cached_property
    def d_f(self) -> float:
        if self.d_f_opt is not None: return float(self.d_f_opt)
        if self.h_f_opt is not None: return self.disp_frac * float(self.h_f_opt)
        return self.disp_frac * (float(self.zom_f) / self.alpha_m)

    @cached_property
    def d_c(self) -> float:
        if self.d_c_opt is not None: return float(self.d_c_opt)
        if self.h_c_opt is not None: return self.disp_frac * float(self.h_c_opt)
        return self.disp_frac * (float(self.zom_c) / self.alpha_m)

    #  zmin respects (z - d) > z0 → keep grid valid for both patches
    @cached_property
    def zmin(self) -> float:
        thresh_f = self.d_f + self.zom_f
        thresh_c = self.d_c + self.zom_c
        return max(thresh_f, thresh_c)   # conservative global lower bound

    @cached_property
    def z(self) -> np.ndarray:
        return np.arange(self.zmin + self.dz, self.Hmax + self.dz, self.dz)

    @cached_property
    def x(self) -> np.ndarray:
        return np.arange(self.xmin, self.Lx + self.dx, self.dx)

    @property
    def nz(self) -> int: return len(self.z)

    @property
    def nx(self) -> int: return len(self.x)

    # u* optional compute (unchanged)
    def resolve_ustars(self):
        if self.ustar_f is None:
            assert self.z_ref_wind > self.zom_f > 0
            self.ustar_f = self.k * self.U_ref / np.log(self.z_ref_wind / self.zom_f)
        if self.ustar_c is None:
            assert self.z_ref_wind > self.zom_c > 0
            self.ustar_c = self.k * self.U_ref / np.log(self.z_ref_wind / self.zom_c)
        return self

    # block sizes (FIXED ternary line break)
    @cached_property
    def fallow_size(self) -> int:
        return int(self.fallow_length / self.dx)

    @cached_property
    def field_size(self) -> int:
        return (
            int(self.fallow_size * (1 - self.fallow_fraction) / self.fallow_fraction)
            if self.fallow_fraction > 0 else 0
        )

    # humidity as absolute humidity (your Q_* names retained)
    @cached_property
    def es_c(self) -> float:
        return saturation_vapor_pressure(self._K(self.T_sc))

    @cached_property
    def es_f(self) -> float:
        return saturation_vapor_pressure(self._K(self.T_sf))

    @cached_property
    def Q_c(self) -> float:
        return vapor_concentration_RH(self.T_sc, self.RH_c)

    @cached_property
    def Q_f(self) -> float:
        return vapor_concentration_RH(self.T_sf, self.RH_f)

    @cached_property
    def Q_a(self) -> float:
        return vapor_concentration_RH(self.T_a, self.RH_a)

    # energy partition
    #  upstream available energy  
    @property
    def RNmG_up(self) -> float:
        return self.H_f

    @property
    def RNmG_down(self) -> float:
        """Downstream available energy = avail_ratio * RNmG_up (W m^-2)."""
        return self.RNmG_up * self.avail_ratio    

    # downstream partition 
    @property
    def LE_c(self) -> float:
        lf = min(max(self.le_factor, 0.0), 1.0)
        return lf * self.RNmG_up * self.avail_ratio

    @property
    def H_c(self) -> float:
        lf = min(max(self.le_factor, 0.0), 1.0)
        return (1.0 - lf) * self.RNmG_up * self.avail_ratio

    # optional dict for legacy call sites
    def to_dict(self) -> dict:
        d = asdict(self)
        d.update({
            "x": self.x, "z": self.z, "nx": self.nx, "nz": self.nz,
            "zmin": self.zmin, "zmax": self.Hmax, "xmax": self.Lx,
            "ustar_f": self.ustar_f, "ustar_c": self.ustar_c,
            "fallow_size": self.fallow_size, "field_size": self.field_size,
            "Q_c": self.Q_c, "Q_f": self.Q_f, "Q_a": self.Q_a,
            "es_c": self.es_c, "es_f": self.es_f,
            # partition values
            "LE_c": self.LE_c, 
            "H_c": self.H_c,
            "RNmG_up": self.RNmG_up, 
            "RNmG_down": self.RNmG_down,
            # canopy/displacement (derived)
            "h_f": self.h_f, "h_c": self.h_c, "d_f": self.d_f, "d_c": self.d_c,
        })
        return d
    
    



    def solve_surface_radiation_inplace(self, *, fix: str = "alpha_f") -> dict:
        """
        Solve the two-surface net-radiation balances using ONLY internal Params state,
        then update Params in place (SW_in and the free albedo).

          Rn_f = SW*(1 - alpha_f) + eps_a*sigma*T_a^4 - eps_f*sigma*T_sf^4
          Rn_c = SW*(1 - alpha_c) + eps_a*sigma*T_a^4 - eps_c*sigma*T_sc^4

        Rn_f = RNmG_up + G;  Rn_c = RNmG_down + G

        Args
        ----
        fix : {"alpha_f","alpha_c"}
            Which albedo to treat as fixed (kept from Params). The other is solved.

        Returns
        -------
        dict with {"SW": float, "alpha_f": float, "alpha_c": float, "Rn_f": float, "Rn_c": float}
        """
        if fix not in ("alpha_f", "alpha_c"):
            raise ValueError("fix must be 'alpha_f' or 'alpha_c'.")

        # Upstream/downstream net radiation from internal partition + ground flux
        Rn_f = float(self.RNmG_up + self.G)
        Rn_c = float(self.RNmG_down + self.G)

        # Temps (°C -> K)
        TaK  = self._K(self.T_a)
        TsfK = self._K(self.T_sf)
        TscK = self._K(self.T_sc)

        # Symbols and constants
        SW = sp.symbols('SW', real=True)
        sigma = sigma_sb
        ea, ef, ec = self.epsilon_a, self.epsilon_f, self.epsilon_c

        if fix == "alpha_f":
            # Unknowns: SW, alpha_c  (alpha_f fixed from Params)
            alpha_c_sym = sp.symbols('alpha_c', real=True)
            eq1 = sp.Eq(SW*(1 - self.alpha_f) + ea*sigma*TaK**4 - ef*sigma*TsfK**4, Rn_f)
            eq2 = sp.Eq(SW*(1 - alpha_c_sym) + ea*sigma*TaK**4 - ec*sigma*TscK**4, Rn_c)
            sol = sp.solve((eq1, eq2), (SW, alpha_c_sym), dict=True)
            if not sol:
                raise RuntimeError("No solution for (SW, alpha_c). Check inputs.")
            SW_val      = float(sol[0][SW])
            alpha_c_val = float(sol[0][alpha_c_sym])

            # Clip to physical range [0,1] with a gentle nudge/warn if needed
            alpha_c_val = min(max(alpha_c_val, 0.0), 1.0)

            # update params in place
            self.SW_in  = SW_val
            self.alpha_c = alpha_c_val

            return {"SW": SW_val, "alpha_f": float(self.alpha_f), "alpha_c": alpha_c_val,
                    "Rn_f": Rn_f, "Rn_c": Rn_c}

        else:
            # fix == "alpha_c": Unknowns: SW, alpha_f  (alpha_c fixed from Params)
            alpha_f_sym = sp.symbols('alpha_f', real=True)
            eq1 = sp.Eq(SW*(1 - alpha_f_sym) + ea*sigma*TaK**4 - ef*sigma*TsfK**4, Rn_f)
            eq2 = sp.Eq(SW*(1 - self.alpha_c) + ea*sigma*TaK**4 - ec*sigma*TscK**4, Rn_c)
            sol = sp.solve((eq1, eq2), (SW, alpha_f_sym), dict=True)
            if not sol:
                raise RuntimeError("No solution for (SW, alpha_f). Check inputs.")
            SW_val      = float(sol[0][SW])
            alpha_f_val = float(sol[0][alpha_f_sym])

            alpha_f_val = min(max(alpha_f_val, 0.0), 1.0)

            # update params in place
            self.SW_in   = SW_val
            self.alpha_f = alpha_f_val

            return {"SW": SW_val, "alpha_f": alpha_f_val, "alpha_c": float(self.alpha_c),
                    "Rn_f": Rn_f, "Rn_c": Rn_c}

p = Params(ustar_f = ustar_f, ustar_c = ustar_f*np.sqrt(shear_ratio),  avail_ratio = 1.2)
out = p.solve_surface_radiation_inplace(fix="alpha_f")


print(p.LE_c)
print (p.H_f)
# Export all params to globals for use in subsequent cells
_pdict = p.to_dict()
for _k, _v in _pdict.items():
    globals()[_k] = _v
# Convert temperatures to Kelvin for downstream cells
T_a = T_a + 273.15
T_sf = T_sf + 273.15
T_sc = T_sc + 273.15
# Extract Rn_f, Rn_c from radiation solution
Rn_f = out['Rn_f']
Rn_c = out['Rn_c']
# Extract derived physics variables from Params object
H_c = p.H_c
H_f = p.H_f
LE_c = p.LE_c
LE_f = p.LE_f
G = p.G
Q_a = p.Q_a
Q_c = p.Q_c
Q_f = p.Q_f
Lv_g = 2430.0  # J/g, latent heat of vaporization


426.768
355.64


In [9]:
from sympy import symbols, Eq, solve
# # downwind humidity is 60%
# G  = 40
# H_f = 356
# LE_f = 0
# Rn_f = H_f + G
# Rn_c = (H_f*1.2 + G)

# H_c  = 0 
# LE_c = Rn_c - G - H_c


# T_a = 30 + 273.15
# T_sc = 30 + 273.15
# Delta_T = 21 
# T_sf = T_sc + Delta_T

# Delta_Q = 8
# Q_a = 6
# Q_f = Q_a
# Q_c =  Q_f + Delta_Q


params = Params(ustar_f = ustar_f, ustar_c = ustar_f*np.sqrt(shear_ratio) ).to_dict()

# Add missing keys expected by downstream code
params.setdefault('h', 2.3)  # canopy height
params.setdefault('adj_grid', 10)  # grid adjustment
params.setdefault('xmax', params.get('Lx', 2000.0))
params.setdefault('zmax', params.get('Hmax', 100.0))
params.setdefault('zmin', params['dz'])

epsilon_a = params['epsilon_a']
epsilon_f = params['epsilon_f']
epsilon_c = params['epsilon_c']


z_h = params['Hmax']
zom_f = params['zom_f']
zom_c = params['zom_c']

k = params['k']
ustar_f = params['ustar_f']
ustar_c = params['ustar_c']

z = np.arange(params['zmin'] + params['dz'], params['zmax'] + params['dz'], params['dz'])

# Define symbols
sigma_sb = 5.67*1e-8
alpha_f = 0.25
# T_sf, T_sc, alpha_f, alpha_c, H_c, LE_c = symbols('T_sf T_sc alpha_f alpha_c H_c  LE_c')
SW, alpha_c = symbols('SW  alpha_c')

level = 0
# unknowns – alpha_f, T_sf
eq1 = Eq(SW*(1-alpha_f) + epsilon_a*sigma_sb*T_a**4 - epsilon_f*sigma_sb*T_sf**4, Rn_f )
eq2 = Eq(SW*(1-alpha_c) + epsilon_a*sigma_sb*T_a**4 - epsilon_c*sigma_sb*T_sc**4, Rn_c )

# Solve the system of equations
solution = solve((eq1, eq2), (SW, alpha_f, alpha_c))


SW, alpha_f, alpha_c = solution[0]
print (SW, alpha_f, alpha_c)


705.357810023631 0.250000000000000 0.293127849702444


In [10]:
# # Define given values
# mcal_to_J = 4.184e-3  # 1 mcal = 4.184e-3 J
# cm2_to_m2 = 1e-4      # 1 cm² = 1e-4 m²
# H01_mcal_cm2_s = 8.5  # Given value in mcal/cm²/s

# # Convert to SI units (W/m²)
# H_f = H01_mcal_cm2_s * mcal_to_J / cm2_to_m2

# LE_f = 0
# # Rn - G = H_f
# Q_f = 6.34 # g mm3, E_01

# RH_c = 60 # relative humidity
# ustar_f = 0.15


Definitions in Rao et al.: 
* $E$ is absolute humidity (mass of water vapor per volume, kg m$^{-3}$); 
* $Q$ is specific humidity (mass of water vapor per mass of moist air, kg kg$^{-1}$).
* Direct relationship: $E = \rho Q$, where $\rho$ is the moist‑air density (kg m$^{-3}$).

* Via vapor pressure: $E = \dfrac{e}{R_v T}$ and $Q = \dfrac{0.622 e}{p - 0.378e}$.
* Combine to get $E = \dfrac{Q p}{(0.622 + 0.378Q) R_v T}$ (with $p$ total pressure, $T$ temperature, $R_v$ the vapor gas constant).

Surface summary: $E_{01}=6.34~\mathrm{g\ m^{-3}}$ and $E_{02}=15.84~\mathrm{g\ m^{-3}}$ 


In [11]:
13.5 - 5.7, 15.8 - 6.3

(7.8, 9.5)

In [12]:
# # ustar_f = 0.15
# # shear_ratio = 2
# # ustar_c = ustar_f*np.sqrt(shear_ratio)
# # print (ustar_c)
 
# # what is h? what is d? 
# from sutton import Params, saturation_vapor_pressure, vapor_concentration_RH
# import numpy as np

# def get_params(fallow_fraction = 0.5, fallow_length = 1000, 
#                T_sc = 30, T_sf = 50, T_a = 18, 
#                Q_a = 5, Q_c = 20, RH_c = 60,
#                h = 0.04, zom_f = 0.14/100, zom_c = 0.14/100,
#                SW_in = 400, e_f = 0.95, e_c = 0.95, e_a = .8,
#                ustar_f = 0.15, ustar_c = ustar_c,
#                alpha_c = 0.22, alpha_f = 0.22,
#                lm_option = 'kz', adj_grid = 0):
#     p = Params(fallow_length=fallow_length, fallow_fraction=fallow_fraction,
#                T_sc=T_sc, T_sf=T_sf, T_a=T_a, RH_c=RH_c, RH_f=RH_c, RH_a=RH_c,
#                h=h, zom_f=zom_f, zom_c=zom_c, SW_in=SW_in, e_f=e_f, e_c=e_c, e_a=e_a,
#                alpha_c=alpha_c, alpha_f=alpha_f, lm_option=lm_option, Hmax=10, dx=0.1, dz=0.001, Lx=30)
#     d = p.to_dict()
#     # Override Q values if provided explicitly
#     d['Q_a'] = Q_a
#     d['Q_c'] = Q_c
#     d['H_f'] = 8.5*4.184e-3/1e-4  # from earlier cell

#     d['adj_grid'] = adj_grid
#     # Legacy counts
#     z = np.arange(d['zmin'] + d['dz'], d['zmax'] + d['dz'], d['dz'])

#     x = np.arange(d['xmin'], d['xmax'] + d['dx'], d['dx'])
#     d['nz'] = len(z)
#     d['nx'] = len(x)
#     return d

# d = get_params()


In [13]:
def uniform_Q(params):
    """
    The function computes an implicit solution for water vapor concentration and
    flux in a turbulent boundary layer.
    
    - Uses an implicit finite-difference method for solving the transport equation.
    - Implements mixing length theory to define eddy diffusivity.
    - Marches along the x-direction to iteratively solve for the concentration field.

    Q_f, Q_c, and Q_a are all absolute humidity (g/m3)
    """
    nx = params['nx']
    nz = params['nz']    
    dx = params['dx']    
    dz = params['dz']  
    zmax = params['zmax']      
    k = params['k']          
    
    ustar_f = params['ustar_f']
    zom_f = params['zom_f']     
    ustar_c = params['ustar_c']
    zom_c = params['zom_c']  
    
    Q_f = params['Q_f'] 
    Q_c = params['Q_c']
    Q_a = params['Q_a']
    
    h = params['h']
    lm_option = params['lm_option']
    adj_grid = params['adj_grid']

    z = np.arange(params['zmin'] + params['dz'], params['zmax'] + params['dz'], 
                  params['dz'])    

    FQ_f = ustar_f*k*(Q_f - Q_a)/np.log(zmax/zom_f)  # g m⁻² s⁻¹
    Qup = Q_f - FQ_f/(k*ustar_f)*np.log(z/zom_f)     # g m⁻³
    
    FQ_c = ustar_c*k*(Q_c - Q_a)/np.log(zmax/zom_c)  # g m⁻² s⁻¹
    Qdown = Q_c - FQ_c/(k*ustar_c)*np.log(z/zom_c)   # g m⁻³
    
    U = (ustar_c / k) * np.log(z / zom_c) 
    
    lm = get_lm(k, h, z, lm_option)    
    
    # Setup coefficients for the implicit scheme
    A = lm * ustar_c
    B = 1.0 / U
    C = our_central_difference(A, dz)

    # Upwind wv concentrations and fluxes
    Q1 = Qup
    Q_uniform = np.zeros((nx , nz))
    Q_uniform[0, :] = Q1
    FluxQ_uniform = np.zeros((nx , nz))

    # Begin downwind calculations by marching along x
    for i in range(nx):
        Q2, Fq = integrate_H2O_implicit(nx, nz, dx, dz, A, B, C, Q1, Qdown[0], Q_a)
        Q_uniform[i, :] = Q2
        FluxQ_uniform[i, :] = Fq
        Q1 = Q2
        
    return Q_uniform, FluxQ_uniform # g m⁻³,  g m⁻² s⁻¹ (mass flux)

# Lv_g = 2430.0  # J g^-1  (~30 °C)
# LE_uniform_Wm2 = FluxQ_uniform * Lv_g     # → W m^-2
# LE_f_Wm2 = FQ_f * Lv_g
# LE_c_Wm2 = FQ_c * Lv_g

def uniform_T(params):
    """
    The function solves an implicit equation for temperature transport in a turbulent boundary layer.
    
    - Uses an implicit finite-difference method to solve the advection-diffusion equation.
    - Mixing length theory is applied to define eddy diffusivity.
    - Marches along the x-direction to iteratively compute the temperature field.
    
    """
    nx = params['nx']
    nz = params['nz']    
    dx = params['dx']        
    dz = params['dz']        
    k = params['k']              
    zmax = params['zmax']
    
    ustar_f = params['ustar_f']
    zom_f = params['zom_f']    
    ustar_c = params['ustar_c']
    zom_c = params['zom_c']  
    
    T_sf = params['T_sf']    
    T_sc = params['T_sc']
    T_a = params['T_a']
    h = params['h']    
    
    lm_option = params['lm_option']
    adj_grid = params['adj_grid']
    
    z = np.arange(params['zmin'] + params['dz'], 
                  params['zmax'] + params['dz'], params['dz'])
    
    wT = - (T_a - T_sf)*k*ustar_f/np.log(zmax/zom_f)      # K m/s 
    Tup = T_sf - wT/(k*ustar_f)*np.log(z/zom_f)

    wT_down = -(T_a - T_sc)*k*ustar_c/np.log(zmax/zom_c)  # K m/s    
    Tdown = T_sc - wT_down/(k*ustar_c)*np.log(z/zom_c)

    U = (ustar_c / k) * np.log(z / zom_c)    
    
    # Setup coefficients for the implicit scheme
    lm = get_lm(k, h, z, lm_option)       
    A = lm * ustar_c
    B = 1.0 / U
    C = our_central_difference(A, dz)

     # Upwind wv concentrations and fluxes
    T1 = Tup
    T_uniform = np.zeros((nx , nz))
    T_uniform[0, :] = T1
    FluxT_uniform = np.zeros((nx , nz))

    for i in range(nx):
        
        T2, FT = integrate_T_implicit(nx, nz, dx, dz, A, B, C, T1, Tdown[0], T_a)
        T_uniform[i, :] = T2
        FluxT_uniform[i, :] = FT
        T1 = T2
    
    return T_uniform, FluxT_uniform

\begin{aligned}
\mathrm{SW}\,(1-\alpha_f) + \varepsilon_a \,\sigma_{SB}\, T_a^{4}
  - \varepsilon_f \,\sigma_{SB}\, T_{sf}^{4} &= R_{n,f}, \\
\mathrm{SW}\,(1-\alpha_c) + \varepsilon_a \,\sigma_{SB}\, T_a^{4}
  - \varepsilon_c \,\sigma_{SB}\, T_{sc}^{4} &= R_{n,c}, \\
T_{sf} - T_a &= \frac{1}{k\,u_{*f}}\,
  \frac{H_f}{\rho\,c_p}\,
  \ln\!\left(\frac{z_h}{z_\ell}\right), \\
T_{sc} - T_a &= \frac{1}{k\,u_{*c}}\,
  \frac{H_c}{\rho\,c_p}\,
  \ln\!\left(\frac{z_h}{z_\ell}\right), \\
Q_c - Q_a &= \frac{1}{k\,u_{*c}}\,
  \frac{\mathrm{LE}_c}{L_v}\,
  \ln\!\left(\frac{z_h}{z_\ell}\right), \\
R_{n,c} - G &= \mathrm{LE}_c + H_c .
\end{aligned}

In [14]:
# T_sf, Q_f are the boundary conditions at level = 5 (top of canopy..?)

# zo = zmin
# ustar_f = 0.15
# ustar_c = 0.15

T_sc =  T_a +  1/(k*ustar_c)*H_c/((1005*1.2))*np.log(z_h/z[level])
T_sf =  T_a +  1/(k*ustar_f)*H_f/(1005*1.2)*np.log(z_h/z[level])

print (T_sc - 273.15)
print (T_sf - 273.15)
print (T_sf - T_sc)
print ("\n")

Q_c =  Q_a +  1/(k*ustar_c)*LE_c/ Lv_g*np.log(z_h/(z[level])) # todo: fix units here
Q_f =  Q_a +  1/(k*ustar_f)*LE_f/ Lv_g*np.log(z_h/(z[level])) # todo: fix units here

print (Q_c)
print (Q_f)
print (Q_c - Q_f)



30.0
54.89384065226534
24.89384065226534


16.084873239942603
5.601544912434969
10.483328327507634


In [15]:
T_zom_f = np.log(z_h/zom_f)/np.log(z_h/z[level])*(T_sf - T_a) + T_a
T_zom_c = np.log(z_h/zom_c)/np.log(z_h/z[level])*(T_sc - T_a) + T_a

Q_zom_f = np.log(z_h/zom_f)/np.log(z_h/z[level])*(Q_f - Q_a) + Q_a
Q_zom_c = np.log(z_h/zom_c)/np.log(z_h/z[level])*(Q_c - Q_a) + Q_a

# print (T_zom_f, T_sf, "\n", T_zom_c, T_sc,"\n", Q_zom_c, alpha_f)

## Log-profile evaluation at the roughness length (neutral case)

**Wet patch (“f”):**
$T_{z_{0m},f} = T_a + \dfrac{\ln\!\left(z_h / z_{0m,f}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(T_{s,f} - T_a\right)$

$Q_{z_{0m},f} = Q_a + \dfrac{\ln\!\left(z_h / z_{0m,f}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(Q_f - Q_a\right)$

**Dry patch (“c”):**
$T_{z_{0m},c} = T_a + \dfrac{\ln\!\left(z_h / z_{0m,c}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(T_{s,c} - T_a\right)$

$Q_{z_{0m},c} = Q_a + \dfrac{\ln\!\left(z_h / z_{0m,c}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(Q_c - Q_a\right)$

---

### Where
- $z_h$ = upper reference (blending) height; $z_{\mathrm{ref}} = z[\text{level}]$ (in-air reference level).  
- $z_{0m,f}, z_{0m,c}$ = **momentum** roughness lengths (wet “f”, dry “c”).  
- $T_a, Q_a$ = air temperature and specific humidity at $z_{\mathrm{ref}}$.  
- $T_{s,f}, T_{s,c}$ = surface (wall) temperatures; $Q_f, Q_c$ = surface (wall) specific humidities.

---


## Log-profile evaluation at the roughness length (neutral case)

**Wet patch (“f”):**
$T_{z_{0m},f} = T_a + \dfrac{\ln\!\left(z_h / z_{0m,f}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(T_{s,f} - T_a\right)$

$Q_{z_{0m},f} = Q_a + \dfrac{\ln\!\left(z_h / z_{0m,f}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(Q_f - Q_a\right)$

**Dry patch (“c”):**
$T_{z_{0m},c} = T_a + \dfrac{\ln\!\left(z_h / z_{0m,c}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(T_{s,c} - T_a\right)$

$Q_{z_{0m},c} = Q_a + \dfrac{\ln\!\left(z_h / z_{0m,c}\right)}{\ln\!\left(z_h / z_{\mathrm{ref}}\right)}\left(Q_c - Q_a\right)$

---

### Where
- $z_h$ = upper reference (blending) height; $z_{\mathrm{ref}} = z[\text{level}]$ (in-air reference level).  
- $z_{0m,f}, z_{0m,c}$ = momentum roughness lengths (wet “f”, dry “c”).  
- $T_a, Q_a$ = air temperature and specific humidity at $z_{\mathrm{ref}}$.  
- $T_{s,f}, T_{s,c}$ = surface (wall) temperatures.
- $Q_f, Q_c$ = surface (wall) specific humidities.



In [16]:
sigma_sb = 5.67*1e-8

# params = get_params(T_sc = float(T_zom_c), T_sf = float(T_zom_f), 
#                     Q_c = float(Q_zom_c), Q_a = Q_a, T_a = T_a, SW_in = SW, 
#                     alpha_f = alpha_f, alpha_c = alpha_c)

In [17]:
Q_a, Q_zom_c

(5.601544912434969, 22.457280445499848)

In [18]:
T_sc, T_sf, T_a

(303.15, 328.0438406522653, 303.15)

In [19]:
LW_out_c = epsilon_c*sigma_sb*(T_sc + 273.15)**4
LW_in_c = epsilon_a*sigma_sb*(T_a + 273.15)**4

Rn_c = (1 - alpha_c)*SW  + LW_in_c - LW_out_c

In [20]:
Rn_c - G, LE_c + H_c

(-439.544676416139, 426.768)

In [21]:
# T_uniform, FluxT_uniform = uniform_T(params)
# Q_uniform, FluxQ_uniform = uniform_Q(params)

# SH = FluxT_uniform*1005*1.2
# LE = FluxQ_uniform*2.5*1e6/1000
# FluxTotal = SH + LE

In [22]:
# plt.plot(x, LE[:, 100])

In [23]:
z[20]

10.885

In [24]:
# FluxT_uniform[-1][level]*1005*1.2, H_c

In [25]:
# FluxQ_uniform[-1][level]*2.5*1e6/1000, LE_c

In [26]:
def get_upwind_fluxQ(params):
    """
    """
    ustar = params['ustar_f']    
    dz = params['dz']
    nz = params['nz']
    k = params['k']
    zmax = params['zmax']
    zoh = params['zom_f']
    h = params['h']
    nx = params['nx']     
    lm_option = params['lm_option']
    adj_grid = params['adj_grid']

    z = np.arange(params['zmin'] + params['dz'],
                  params['zmax'] + params['dz'], params['dz'])
    
    #zc = get_zc(params)
    
    k = params['k']
    Q_f = params['Q_f']
    Q_a = params['Q_a']
    
    LE_f = ustar*k*(Q_f - Q_a)/np.log(zmax/zom_f)

    Qup = Q_f - LE_f/(k*ustar)*np.log(z/zom_f)
    dQdz = our_central_difference(Qup, dz)

    lm = get_lm(k,h, z, lm_option)
    
    A = lm * ustar
    
    FluxQ_upwind = np.zeros_like(Qup)    
    FluxQ_upwind[1:] = - 0.5 * (A[1:] + A[:-1]) * (0*dQdz[:-1] + dQdz[1:]) * 0.5

    FluxQ_upwind[1] = FluxQ_upwind[2]
    FluxQ_upwind[0] = FluxQ_upwind[1] 
    
    # FluxQ_upwind = - A * dQdz
    
    return Qup, FluxQ_upwind

def get_upwind_fluxT(params):
    """
    """
    dz = params['dz']
    nz = params['nz']
    nx = params['nx']    
    ustar = params['ustar_f']
    zmax = params['zmax']
    zoh_f = params['zom_f']/5
    zom_f = params['zom_f']
    h = params['h']
    lm_option = params['lm_option']    
    adj_grid = params['adj_grid']    
    
    z = np.arange(params['zmin'] + params['dz'],
                  params['zmax'] + params['dz'], params['dz'])
    
    k = params['k']
    T_sf = params['T_sf']
    T_a = params['T_a']
    Q_a = params['Q_a']        
    
    wT = - (T_a - T_sf)*k*ustar/np.log(zmax/zom_f) # K m/s
    Tup = T_sf - wT/(k*ustar)*np.log(z/zom_f)
    
    dTdz = our_central_difference(Tup, dz)

    ustar = params['ustar_f']
    k = params['k']

    lm = get_lm(k, h, z, lm_option)    
    A = lm * ustar

    # FluxT_upwind = - A * dTdz
    
    FluxT_upwind = np.zeros_like(Tup)
    FluxT_upwind[1:] = - 0.5 * (A[1:] + A[:-1]) * (0*dTdz[:-1] + dTdz[1:]) * 0.5
    
    FluxT_upwind[1] = FluxT_upwind[2]
    FluxT_upwind[0] = FluxT_upwind[1]
    
    return Tup, FluxT_upwind

Qup, FluxQ_upwind = get_upwind_fluxQ(params)
Tup, FluxT_upwind = get_upwind_fluxT(params)



In [27]:
x = np.arange(params['xmin'], params['xmax'] + params['dx'], params['dx'])

FluxTotal_upwind = FluxQ_upwind[level]*np.ones_like(x)*2.260*1e6/1000 + \
            FluxT_upwind[level]*np.ones_like(x)*1005*1.2

In [28]:

# plt.figure(figsize = (10, 4 ))
# ax = plt.gca()

# x = np.arange(params['xmin'], params['xmax'] + params['dx'], params['dx'])
# xx = np.concatenate([-np.flip(x),x])

# FluxTotal_ = np.concatenate([FluxTotal_upwind, FluxTotal[:, level]])

# FluxTotal_upwind = FluxQ_upwind[level]*np.ones_like(x)*2.260*1e6/1000 + \
#             FluxT_upwind[level]*np.ones_like(x)*1005*1.2

# ax.plot(xx, FluxTotal_,  c =  'C0', ls = '--', label = '$H + LE$')

# FluxQ = np.concatenate([FluxQ_upwind[level]*np.ones_like(x), FluxQ_uniform[:, level]])
# ax.plot(xx,FluxQ*2.260,  ls = '--', c =  'C2',  label = "$LE$")

# FluxT = np.concatenate([FluxT_upwind[level]*np.ones_like(x), FluxT_uniform[:,level]])
# ax.plot(xx, FluxT*1005*1.2,  c =  'C1', ls = '--', label = "$H$")
# ax.legend()
# ax.set_title("Surface sensible and latent heat fluxes (W/m$^2$)")

# level = 6
# print (FluxQ_upwind[level]* Lv_g,  FluxT_upwind[level]*1005*1.2)
# print (FluxT_uniform[ -1, level]*1005*1.2, FluxQ_uniform[ -1, level]* Lv_g)


In [29]:
# FluxQ_uniform

In [30]:
fig6data = pd.read_csv('../../data/baldocchi1995/baldocchi1995_fig6.csv')

In [31]:
# 4 m above the surface
# assume a 1 m canopy
# 3 m above the canopy
# z = height above canopy - d


In [32]:

# i800 = np.where(x > 800)[0][0]
# plt.scatter(fig6data['x'], fig6data[' y'], label = "Baldocchi and Rao, 1995")
# # level = 3
# # plt.plot(x[:i800], FluxQ_uniform[:i800, level]/FluxQ_uniform[i800, level])

# level = 40
# plt.plot(x[:i800], FluxQ_uniform[:i800, level]/FluxQ_uniform[i800, level])
# plt.xlabel("x")
# plt.xlabel("ET/ET(800m)")

In [33]:
# x = np.arange(params['xmin'], params['xmax'] + params['dx'], params['dx'])

# level = 10
# plt.figure(figsize = (10,4))
# ax = plt.gca()
# ax.plot(x[1:], FluxTotal[1:, level], label = 'total')
# ax.plot(x[1:], FluxQ_uniform[1:, level]*2.260*1e6/1000, label = "$LE$")
# ax.plot(x[1:], FluxT_uniform[1:, level]*1005*1.2, label = "$SH$")
# ax.axhline(FluxT_uniform[-1, level], ls = ':', c = 'k')
# ax.legend()
# ax.set_xlim(0, 500)
# ax.set_title("Surface sensible and latent heat flux (W/m$^2$)")

# FluxTotal[level, -1]

In [34]:
# z = np.arange(params['zmin'], params['zmax'] + params['dz'], params['dz'])
# x = np.arange(params['xmin'], params['xmax'] + params['dx'], params['dx'])

# Plot water vapor concentration and vertical flux
# plt.figure(figsize = (8, 6))

# plt.subplot(2, 1, 1)
# nz = Q_uniform.shape[1]
# plt.contourf(x, z[:nz], (Q_uniform.T), 25, cmap='Blues',  vmax = Q_c, vmin = Q_a)
# plt.colorbar(label='Water vapor concentration \n (gm m$^{-3}$)')
# plt.xlabel('x (m)', fontsize=12, fontweight='normal')
# plt.ylabel('z (m)', fontsize=12, fontweight='normal')
# plt.title('Water vapor concentration $q$ (gm m$^{-3}$)', fontsize=12)

# plt.subplot(2, 1, 2)
# plt.contourf(x, z[:nz], FluxQ_uniform.T.round(4),  20, cmap='coolwarm', vmin = 0, vmax = FluxQ_uniform.max())
# plt.colorbar(label='Water vapor flux \n (gm m$^{-2}$ s$^{-1}$)') 
# plt.xlabel('x (m)', fontsize=12, fontweight='normal')
# plt.ylabel('z (m)', fontsize=12, fontweight='normal')
# plt.title('Water vapor flux (gm m$^{-2}$ s$^{-1}$)', fontsize=12)
# plt.tight_layout()




1) Surface energy balance  
   $H_{02}(x) + L\,F_{02}(x) = R_N - G$

The available energy at the surface ($R_N-G$) is partitioned into sensible ($H$) and latent ($LE=L F$) heat fluxes that warm the air and evaporate water, respectively. Over a wetter patch, more of $R_N-G$ typically goes into $LE$, leaving less for $H$ and cooling the surface layer.

2) Surface humidity (RH) boundary condition  
   $Q_0(x) = R_s\,Q_s\!\big(\Theta_0(x)\big)$

The surface air right above the wet patch is assumed to be at a fixed relative humidity $R_s$; thus its specific humidity equals $R_s$ times the saturation value at the (to-be-determined) surface temperature. Because $Q_s$ increases quickly with $\Theta_0$, warmer surfaces push $Q_0$ higher for a given $R_s$.


Together with the surface-layer (Kansas/Monin–Obukhov) flux–profile relations,
$
\frac{\partial \Theta}{\partial z}=-\frac{T_*}{k z}\phi_h,\qquad
\frac{\partial Q}{\partial z}=-\frac{q_*}{k z}\phi_q,
$
and with the star scales
$
T_* = -\frac{\overline{w\theta}}{u_*},\qquad
q_* = -\frac{\overline{wq}}{u_*},\qquad
H = -\rho c_p\,\overline{w\theta},\quad LE = -\rho L\,\overline{wq},
$
the two equations determine, at each fetch $x$, the unknown surface state and fluxes:
$
\{\,\Theta_0(x),\,Q_0(x),\,H_{02}(x),\,F_{02}(x)\,\}
\quad\text{and hence}\quad
\{\,u_*(x),\,T_*(x),\,q_*(x)\,\}.
$

- Hold $R_N-G$ fixed. If the surface is wet, the RH condition pushes $Q_0$ close to saturation. That promotes evaporation ($F_{02}\!\uparrow$), so latent heat rises ($LE=L F_{02}$), leaving less for sensible heat ($H_{02}\!\downarrow$) and usually a cooler $\Theta_0$ relative to upstream dry terrain.


### Appendix

In [35]:
# density depends on air temperature
import numpy as np

# --- constants ---
cp  = 1005.0          # J kg^-1 K^-1  (specific heat at constant pressure)
R_d = 287.05          # J kg^-1 K^-1  (gas constant for dry air)
p_Pa = 101325.0       # Pa (set your measured station pressure if available)

def air_density(T_K, p_Pa=101325.0, q_kgkg=None):
    """
    Air density [kg m^-3] from temperature (K), pressure (Pa), and optional specific humidity q (kg/kg).
    Uses virtual temperature: T_v = T*(1 + 0.61 q) if q is provided.
    """
    if q_kgkg is not None:
        T_v = T_K * (1.0 + 0.61*q_kgkg)
    else:
        T_v = T_K
    return p_Pa / (R_d * T_v)

# If you already have ambient humidity as Q_a in g/kg, use it for moist-air density.
# Otherwise we'll compute a dry-air density.
try:
    q_a = Q_a / 1000.0   # g/kg -> kg/kg
except NameError:
    q_a = None

rho = air_density(T_a, p_Pa=p_Pa, q_kgkg=q_a)

# --- your original variables ---
zo = zom_f
ustar_f = 0.15
ustar_c = 0.15

# stability-neutral bulk transfer (ensure z_h > zo)
L = np.log(z_h / zo)      # dimensionless

# Use rho*cp instead of 1005*1.2
T_sc = T_a + (H_c/(rho*cp)) * (1.0/(k*ustar_c)) * L
print(T_sc - 273.15)

T_sf = T_a + (H_f/(rho*cp)) * (1.0/(k*ustar_f)) * L
print(T_sf - 273.15)

print(T_sf - T_sc)


30.0
80.33399371402442
50.333993714024416


### Appendix B - solve all BCs

In [36]:
# from sympy import symbols, Eq, solve
# downwind humidity is 60%
# G  = 30
# Rn_f = H_f + G
# Rn_c = H_f + G

# SW = 500
# T_a = 30 + 273.15
# Q_a = 6

# epsilon_a = 0.95
# epsilon_f = 0.8
# epsilon_c = 0.95

# params = get_params(h = 2.3, SW_in = SW, 
#                e_f = epsilon_f, e_c = epsilon_c, e_a = epsilon_a,            
#                lm_option = 'kz', adj_grid = 0)

# z_h = params['Hmax']

# zom_f = params['zom_f']
# zom_c = params['zom_c']
# zoh_f = params['zom_f']/5
# zoh_c = params['zom_c']/5

# k = params['k']
# ustar_f = params['ustar_f']
# ustar_c = params['ustar_c']

# z = np.arange(params['zmin'] + params['dz'], params['zmax'] + params['dz'], params['dz'])

# Define symbols
# sigma_sb = 5.67*1e-8

# Q_f = Q_a
# Q_c =  Q_f + 8

# T_sf, T_sc, alpha_f, alpha_c, H_c, LE_c = symbols('T_sf T_sc alpha_f alpha_c H_c  LE_c')

# level = 0

# eq1 = Eq(SW*(1-alpha_f) + epsilon_a*sigma_sb*T_a**4 - epsilon_f*sigma_sb*T_sf**4, Rn_f )

# eq2 = Eq(SW*(1-alpha_c) + epsilon_a*sigma_sb*T_a**4 - epsilon_c*sigma_sb*T_sc**4, Rn_c )

# eq3 = Eq(T_sf - T_a, 1/(k*ustar_f)*H_f/((1005*1.2))*np.log(z_h/(z[level])))

# eq4 = Eq(T_sc - T_a, 1/(k*ustar_c)*H_c/((1005*1.2))*np.log(z_h/(z[level])))

# eq5 = Eq(Q_c - Q_a, 1/(k*ustar_c)*LE_c/ Lv_g*np.log(z_h/(z[level])))

# eq6 = Eq(Rn_c - G, LE_c +  H_c)

# Solve the system of equations
# solution = solve((eq1, eq2, eq3,  eq4, eq5, eq6), 
#                  (T_sf, T_sc, alpha_f, alpha_c, LE_c,  H_c ))

# T_sf, T_sc, alpha_f, alpha_c, LE_c,  H_c = solution[0]

# T_a = T_a - 273.15
# T_sf = T_sf - 273.15
# T_sc = T_sc - 273.15

# T_sf,  T_sc